<a href="https://colab.research.google.com/github/calliemariah/Energetic-Koala-CS-374-Algorithms-Project-Code/blob/main/CS374_EnergeticKoalas_AlgorithmsProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Graph for Facebook Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

#Coloring Section


# Initializing FaceBook Data Frame

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Define the file path
file_path = '/content/drive/MyDrive/pseudo_facebook.csv'

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)

# Display the first 5 rows of the DataFrame
display(df.head())

# Coloring Graphs for Facebook Data



In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

# Drop rows with missing 'tenure' or 'age' as they are critical for graph construction
df_filtered = df.dropna(subset=['tenure', 'age'])

# 1. Define nodes: Unique tenure values
tenure_nodes = sorted(df_filtered['tenure'].unique().tolist())

# Create an empty graph
G = nx.Graph()
G.add_nodes_from(tenure_nodes)

# 2. Define edges: An edge exists between two distinct tenure values
#    if there is at least one age group that contains users from both tenures.
#    This implies a conflict between these tenures (mediated by users sharing an age group).

# Group users by age and collect the unique tenures present in each age group
age_to_tenures = df_filtered.groupby('age')['tenure'].apply(lambda x: list(x.unique())).to_dict()

edges_to_add = set()

# Iterate through each age group
for age, tenures_in_age in age_to_tenures.items():
    # If an age group contains multiple unique tenure values,
    # then these tenure values conflict with each other within this age group.
    if len(tenures_in_age) > 1:
        # Create edges between all pairs of unique tenures found in this age group
        for i in range(len(tenures_in_age)):
            for j in range(i + 1, len(tenures_in_age)):
                t1 = tenures_in_age[i]
                t2 = tenures_in_age[j]
                # Add the edge (sorted tuple to avoid duplicates like (a,b) and (b,a))
                edges_to_add.add(tuple(sorted((t1, t2))))

G.add_edges_from(list(edges_to_add))

# Perform graph coloring using a greedy algorithm
# The 'largest_first' strategy colors nodes with higher degrees first.
coloring = nx.coloring.greedy_color(G, strategy='largest_first')

num_colors = len(set(coloring.values()))

# Prepare for visualization
# Assign a unique color from a colormap to each distinct color ID used by the coloring algorithm
cmap = plt.get_cmap('tab20', max(coloring.values()) + 1 if num_colors > 0 else 1) # 'tab20' for distinct colors
node_colors = [cmap(coloring[node]) for node in G.nodes()]

plt.figure(figsize=(14, 12))
# Use spring_layout for better visualization of connections
pos = nx.spring_layout(G, k=0.8, iterations=50) # Adjust k and iterations for layout
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=800, alpha=0.9)
nx.draw_networkx_edges(G, pos, edge_color='gray', alpha=0.4, width=0.5)
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold')
plt.title(f"Graph Coloring of Tenure Nodes (Conflicts from Users Sharing Age Group)\n{num_colors} Colors Used", fontsize=16)
plt.axis('off')

# Create a legend for the colors
# Mapping actual color IDs to indices for the colormap
unique_color_ids = sorted(list(set(coloring.values())))
legend_patches = [plt.Line2D([0], [0], marker='o', color='w', label=f'Color {c_id}',
                             markerfacecolor=cmap(c_id), markersize=10)
                  for c_id in unique_color_ids]
plt.legend(handles=legend_patches, title="Assigned Colors", bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10, title_fontsize='12')

plt.tight_layout()
plt.show()

print(f"Number of tenure nodes: {G.number_of_nodes()}")
print(f"Number of edges (conflicts defined by shared age groups): {G.number_of_edges()}")
print(f"Number of colors used for coloring: {num_colors}")
print("\nNode coloring (tenure: assigned_color_id):")
# Print coloring results in a readable format, sorted by tenure
for tenure, color_id in sorted(coloring.items(), key=lambda item: item[0]):
    print(f"  Tenure {int(tenure)}: Color {color_id}")

### Visualizing Similarities



In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Ensure df_filtered is available and cleaned from previous steps
# (Already done in cell tCjjEHM1F1kw, assuming it ran successfully)

# 1. Create a matrix of tenures vs. age groups
# For each unique tenure, create a vector indicating presence in each age group.
# We use `groupby` and then `unstack` to create this matrix.

# Get all unique age values to ensure consistent columns in the matrix
all_ages = sorted(df_filtered['age'].unique())

# Create a pivot table: index=tenure, columns=age, values=1 for presence, 0 for absence
tenure_age_matrix = df_filtered.groupby('tenure')['age'].apply(lambda x: pd.Series(1, index=x.unique()))

# Reindex to ensure all ages are present as columns and fill NaNs with 0 (meaning absence)
tenure_age_matrix = tenure_age_matrix.unstack(fill_value=0)

# If a tenure has no associated age (should not happen after dropna, but for robustness)
tenure_age_matrix = tenure_age_matrix.loc[df_filtered['tenure'].dropna().unique()]

# Filter out ages that might not have any representation after unstacking
# This ensures the matrix is clean for t-SNE
tenure_age_matrix = tenure_age_matrix.loc[:, (tenure_age_matrix != 0).any(axis=0)]

print(f"Shape of Tenure-Age Matrix: {tenure_age_matrix.shape}")

# 2. Apply t-SNE to reduce dimensionality
# Perplexity is a critical parameter, usually between 5 and 50.
# We choose a value considering the number of data points. If too many points, try higher.
# n_iter_without_progress helps to stop early if convergence is slow.

# Adjust perplexity based on the number of samples (tenures)
# Rule of thumb: 5 < perplexity < N-1 where N is number of samples
# If N is small, perplexity should be small.
# For large N (2426 tenures), 30 or 50 is common.
perplexity_value = min(50, tenure_age_matrix.shape[0] - 1) if tenure_age_matrix.shape[0] > 1 else 1

tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity_value, n_iter_without_progress=300)
tenure_embeddings = tsne.fit_transform(tenure_age_matrix)

# Create a DataFrame for easier plotting
tenure_embeddings_df = pd.DataFrame(tenure_embeddings, columns=['tSNE1', 'tSNE2'], index=tenure_age_matrix.index)

print("t-SNE embedding complete.")


### Modified Coloring Visualization



In [ ]:
import seaborn as sns

plt.figure(figsize=(12, 10))
sns.scatterplot(
    x='tSNE1', y='tSNE2',
    data=tenure_embeddings_df,
    hue=tenure_embeddings_df.index.to_series().astype(int), # Use tenure itself for coloring, convert to int for better hue mapping
    palette='viridis', # Choose a color palette
    legend='full',
    alpha=0.7,
    s=50 # Point size
)

plt.title('t-SNE Visualization of Tenure Similarities (based on Age Group Presence)', fontsize=16)
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

# Make the legend less dense for many colors (optional, for readability)
num_unique_tenures = len(tenure_embeddings_df.index)
if num_unique_tenures > 20:
    # Reduce the number of legend items by showing only a subset or a color bar
    # For simplicity, we can remove the legend completely if too many, or just show a colorbar if applicable.
    # For this many points, a color bar is better or no legend at all.
    plt.gca().get_legend().remove() # Remove the legend as it would be too dense
    print("Legend removed due to too many unique tenure values for better plot readability.")

plt.tight_layout()
plt.show()

### Clustering Tenures



In [ ]:
from sklearn.cluster import KMeans
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)

# Choosing an arbitrary number of clusters (k=5) for demonstration
# In a real scenario, you might use methods like the Elbow method or Silhouette score
# to determine an optimal k.
k_clusters = 5

kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init=10) # n_init explicitly set for KMeans
tenure_embeddings_df['cluster'] = kmeans.fit_predict(tenure_embeddings_df[['tSNE1', 'tSNE2']])

print(f"K-Means clustering applied with {k_clusters} clusters.")
display(tenure_embeddings_df.head())

### Visualizing the clusters

In [ ]:
import seaborn as sns

plt.figure(figsize=(12, 10))
sns.scatterplot(
    x='tSNE1', y='tSNE2',
    data=tenure_embeddings_df,
    hue='cluster', # Color by cluster ID
    palette='tab10', # Use a distinct palette for clusters
    legend='full',
    alpha=0.7,
    s=50 # Point size
)

plt.title(f't-SNE Visualization with {k_clusters} K-Means Clusters', fontsize=16)
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### Average Tenure per Cluster



In [ ]:
tenure_to_cluster = tenure_embeddings_df['cluster'].to_dict()

df_clustered = df_filtered.copy()
df_clustered['cluster'] = df_clustered['tenure'].map(tenure_to_cluster)

df_clustered = df_clustered.dropna(subset=['cluster'])
df_clustered['cluster'] = df_clustered['cluster'].astype(int)

average_tenure_per_cluster = df_clustered.groupby('cluster')['tenure'].mean().reset_index()
average_tenure_per_cluster['tenure'] = average_tenure_per_cluster['tenure'].round(2)

print("Average Tenure (in days) for Each Cluster:")
display(average_tenure_per_cluster)

### Tenure Variation by Cluster (Box Plot)



In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='cluster', y='tenure', data=df_clustered, palette='viridis')
plt.title('Tenure Variation by Cluster')
plt.xlabel('Cluster ID')
plt.ylabel('Tenure (days)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Merge cluster information back to the original filtered dataframe (or tenure_age_matrix)
# First, we need to map the cluster labels to the original tenures.

# Create a mapping from tenure to cluster
tenure_to_cluster = tenure_embeddings_df['cluster'].to_dict()

# Add cluster information to df_filtered.
# Since df_filtered can have multiple rows per tenure (for different users),
# we map the cluster to each user's tenure.
# We need to handle potential NaN tenures that might have been dropped,
# but df_filtered is already subsetted to non-null tenures.
df_clustered = df_filtered.copy()
df_clustered['cluster'] = df_clustered['tenure'].map(tenure_to_cluster)

# Filter out any rows where cluster might be NaN if a tenure from df_filtered was not in tenure_embeddings_df (shouldn't happen here)
df_clustered = df_clustered.dropna(subset=['cluster'])
df_clustered['cluster'] = df_clustered['cluster'].astype(int)


print("Most common age groups per cluster:")
print("----------------------------------")

# Analyze age distribution for each cluster
for cluster_id in sorted(df_clustered['cluster'].unique()):
    cluster_df = df_clustered[df_clustered['cluster'] == cluster_id]

    # Get the value counts for 'age' within this cluster
    age_counts = cluster_df['age'].value_counts(normalize=True).head(5) # Top 5 most common ages

    print(f"\nCluster {cluster_id}:")
    if not age_counts.empty:
        for age, proportion in age_counts.items():
            print(f"  Age {int(age)}: {proportion:.2%}")
    else:
        print("  No age data for this cluster.")


# Optional: Visualize the age distribution across clusters (e.g., as bar plots)
plt.figure(figsize=(15, 8))
selected_ages = df_clustered['age'].value_counts().head(10).index # Top 10 overall ages

sns.countplot(
    data=df_clustered[df_clustered['age'].isin(selected_ages)],
    x='age',
    hue='cluster',
    palette='tab10',
    order=selected_ages
)
plt.title('Distribution of Top 10 Age Groups Across K-Means Clusters')
plt.xlabel('Age Group')
plt.ylabel('Number of Users')
plt.xticks(rotation=45)
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()


# BFS Section


In [ ]:
FB_Kaggle = pd.read_csv("/content/drive/MyDrive/pseudo_facebook.csv")
df = FB_Kaggle.sample(500)

In [ ]:
from collections import deque
import random

In [ ]:
def similarity(row_i, row_j):
    #similarity score
    score = 0

    #skip first column (user id), compare all other attributes
    for col in row_i.index[1:]:
        val_i = row_i[col]
        val_j = row_j[col]

        #if numeric values, treat as similar if they are close
        if isinstance(val_i, (int, float)) and isinstance(val_j, (int, float)):
            if abs(val_i - val_j) <= 2:
                score += 1

        #if categorical values, must match exactly
        else:
            if val_i == val_j:
                score += 1

    return score

In [ ]:
def build_graph(df, threshold):
    #nodes = users, edges = similarity
    graph = {}

    #initialize empty adjacency list for each user
    for i in range(len(df)):
        user_i = df.iloc[i]["userid"]
        graph[user_i] = []

    #compare every pair of users
    for i in range(len(df)):
        for j in range(i + 1, len(df)):
            row_i = df.iloc[i]
            row_j = df.iloc[j]

            sim = similarity(row_i, row_j)

            #create edge if similarity is above threshold
            if sim >= threshold:
                user_i = row_i["userid"]
                user_j = row_j["userid"]
                graph[user_i].append(user_j)
                graph[user_j].append(user_i)

    return graph

In [ ]:
def bfs(graph, start):
    queue = deque([start])
    visited = set([start])

    #store shortest distance from start node
    distance = {start: 0}

    #track how many nodes are found at each distance level
    level_count = {0: 1}

    while queue:
        node = queue.popleft()
        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

                #distance = parent's distance + 1
                d = distance[node] + 1
                distance[neighbor] = d

                level_count[d] = level_count.get(d, 0) + 1

    return distance, level_count

In [ ]:
def analyze_graph(graph):
    degrees = [len(graph[node]) for node in graph]
    avg_degree = sum(degrees) / len(degrees)

    return {
        "nodes": len(graph),          #total num of users
        "avg_degree": avg_degree,     #avg num of connections per user
        "max_degree": max(degrees)    #most connected user
    }

In [ ]:
def analyze_spread(graph, num_trials=10):
    nodes = list(graph.keys())

    #only start from nodes with at least one connection
    valid_nodes = [n for n in nodes if len(graph[n]) > 0]

    #randomly choose starting nodes
    start_nodes = random.sample(valid_nodes,
                                min(num_trials, len(valid_nodes)))

    results = []

    for start in start_nodes:
        dist, levels = bfs(graph, start)
        total = len(dist)
        avg_dist = sum(dist.values()) / total
        max_dist = max(dist.values())
        cumulative = 0
        step_80 = None

        #how many steps it takes to reach 80% of nodes
        for level in sorted(levels):
            cumulative += levels[level]
            if step_80 is None and cumulative / total >= 0.8:
                step_80 = level

        results.append({
            "reachable": total,
            "avg_dist": avg_dist,
            "max_dist": max_dist,
            "step_80": step_80
        })

    return results

In [ ]:
def largest_component(graph):
    visited = set()
    largest = 0

    for node in graph:
        if node not in visited:
            dist, _ = bfs(graph, node)
            component_size = len(dist)

            #mark all nodes in this component as visited
            visited.update(dist.keys())

            #track largest component seen so far
            largest = max(largest, component_size)

    return largest

In [ ]:
def summarize(threshold, graph_stats, spread_results):
    avg_reach = sum(r["reachable"] for r in spread_results) / len(spread_results)
    avg_dist = sum(r["avg_dist"] for r in spread_results) / len(spread_results)
    avg_max = sum(r["max_dist"] for r in spread_results) / len(spread_results)
    avg_step80 = sum(r["step_80"] for r in spread_results) / len(spread_results)

    print("\n--------------------------------------------")
    print(f"SIMILARITY THRESHOLD = {threshold}")
    print("--------------------------------------------")

    print("\nGRAPH STRUCTURE")
    print("-------------------")
    print("Nodes:", graph_stats["nodes"])
    print("Average Degree:", round(graph_stats["avg_degree"], 2))
    print("Max Degree:", graph_stats["max_degree"])

    print("\nTRAVERSAL RESULTS")
    print("-------------------")
    print("Average Reachable Nodes:", round(avg_reach, 2))
    print("Average Distance:", round(avg_dist, 2))
    print("Average Max Distance:", round(avg_max, 2))
    print("Average Step to Reach 80%:", round(avg_step80, 2))

In [ ]:
df = FB_Kaggle.sample(500)
thresholds = [3, 5, 7, 9]

reachable = []
avg_step80 = []

for t in thresholds:
    graph = build_graph(df, t)
    graph_stats = analyze_graph(graph)
    spread_results = analyze_spread(graph, num_trials=20)
    summarize(t, graph_stats, spread_results)
    avg_reach = sum(r["reachable"] for r in spread_results) / len(spread_results)
    reachable.append(avg_reach)
    steps = [r["step_80"] for r in spread_results if r["step_80"] is not None]
    avg_step = sum(steps) / len(steps)
    avg_step80.append(avg_step)

    print("Largest Connected Component:", largest_component(graph))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,5))
plt.plot(thresholds, reachable, marker='o')
plt.xlabel("Similarity Threshold")
plt.ylabel("Average Reachable Nodes")
plt.title("Threshold vs Network Reachability")
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(thresholds, avg_step80, marker='o')
plt.xlabel("Similarity Threshold")
plt.ylabel("Average Steps to Reach 80%")
plt.title("Threshold vs Information Spread Speed")
plt.grid(True)

plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import random

df = FB_Kaggle.sample(500)
thresholds = [3, 5, 7, 9]

graphs = {}

for t in thresholds:
    graphs[t] = build_graph(df, t)

    G = nx.Graph()
    for node, neighbors in graph.items():
        for n in neighbors:
            G.add_edge(node, n)

    #small subgraph
    small_nodes = random.sample(list(G.nodes), min(30, len(G.nodes)))
    SG = G.subgraph(small_nodes)

    plt.figure(figsize=(8,6))
    pos = nx.spring_layout(SG, seed=42)

    nx.draw(SG, pos, with_labels=False, node_size=200)

    plt.title("Sample Subgraph of Social Network (Threshold = " + str(t) + ")")
    plt.show()

# DFS Graphing

In [ ]:
# Different set up with subset:
# Create an empty graph
G = nx.Graph()

# Limit data and add nodes
df_subset = df.head(100)
for index, row in df_subset.iterrows():
    G.add_node(row['userid'], tenure=row['tenure'], age=row['age'], gender=row['gender'])

# Group by age and gender
for (age, gender), group in df_subset.groupby(['age', 'gender']):
    users_in_group = group['userid'].tolist()

    if len(users_in_group) > 1:
        # We don't want completely interconnected edges within clusters, so we choose one as the one that interconnects all the others:
        hub_node = users_in_group[0]

        # Connect this new 'hub' to every other user in the group
        # This discards edges between the children of the hub.
        for other_user in users_in_group[1:]:
            G.add_edge(hub_node, other_user)


# Add some "bridge" relations, such as those with the same Birth Year
# Allows DFS to jump from an 'Age 20/Male' cluster to an 'Age 20/Female' cluster in similarity
for dob_year, group in df_subset.groupby('dob_year'):
    users = group['userid'].tolist()
    if len(users) > 1:
        # Just connect the first person of this year to the last person to create a sparse bridge between their age/gender clusters.
        G.add_edge(users[0], users[-1])


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
pos = nx.spring_layout(G) # positions for all nodes
nx.draw_networkx_nodes(G, pos, node_size=20, node_color='blue')
nx.draw_networkx_edges(G, pos, alpha=0.5, edge_color='gray')
plt.title('Graph Visualization of Users by Age and Gender')
plt.axis('off') # Hide axes
plt.show()

In [ ]:
# Find all connected components in the graph
connected_components = list(nx.connected_components(G))

print(f"Found {len(connected_components)} connected components.")

# Select one component to focus on (e.g., the largest one)
if connected_components:
    main_component_nodes = max(connected_components, key=len)
    selected_cluster_graph = G.subgraph(main_component_nodes).copy() # Use .copy() to create a new graph
    print(f"Selected a cluster with {selected_cluster_graph.number_of_nodes()} nodes and {selected_cluster_graph.number_of_edges()} edges.")
else:
    print("No connected components found in the graph.")
    selected_cluster_graph = nx.Graph() # Create an empty graph if no components

# Choose a starting node for DFS (e.g., the first node in the selected cluster)
if selected_cluster_graph.nodes():
    start_node = list(selected_cluster_graph.nodes())[0]
    print(f"Starting DFS from node: {start_node}")
else:
    start_node = None
    print("No nodes in the selected cluster to start DFS.")

In [ ]:
if start_node:
    # Perform DFS and keep track of visited nodes and their tenures
    dfs_nodes = []
    dfs_edges = []
    max_tenure = -1
    user_with_longest_tenure = None

    # Using nx.dfs_edges to get the edges in DFS order
    for u, v in nx.dfs_edges(selected_cluster_graph, start_node):
        last_node = v


        dfs_nodes.append(u)
        dfs_nodes.append(v)
        dfs_edges.append((u, v))

        # Check tenure of node u
        current_tenure_u = selected_cluster_graph.nodes[u].get('tenure', 0)
        if current_tenure_u > max_tenure:
            max_tenure = current_tenure_u
            user_with_longest_tenure = u

        # Check tenure of node v
        current_tenure_v = selected_cluster_graph.nodes[v].get('tenure', 0)
        if current_tenure_v > max_tenure:
            max_tenure = current_tenure_v
            user_with_longest_tenure = v

        if start_node:
        # Dictionary to store distance from start_node
        # Using shortest_path_length ensures we know the 'minimum' indirect distance
          distances = nx.single_source_shortest_path_length(selected_cluster_graph, start_node)

          indirect_friends = [node for node, dist in distances.items() if dist > 1]
          direct_friends = [node for node, dist in distances.items() if dist == 1]

          print(f"User: {start_node}")
          print(f"Direct friends: {len(direct_friends)}")
          print(f"Indirect relations found: {len(indirect_friends)}")

          # If you must use DFS order to find them:
          dfs_successors = list(nx.dfs_successors(selected_cluster_graph, start_node))
          # This shows the hierarchy of the social chain


    # Add the start node itself to dfs_nodes if it wasn't added as an endpoint of an edge
    if start_node not in dfs_nodes:
        dfs_nodes.append(start_node)
        current_tenure_start = selected_cluster_graph.nodes[start_node].get('tenure', 0)
        if current_tenure_start > max_tenure:
            max_tenure = current_tenure_start
            user_with_longest_tenure = start_node

    # Ensure dfs_nodes contains unique nodes encountered during DFS
    dfs_nodes = list(set(dfs_nodes))

    print(f"DFS performed on {len(dfs_nodes)} nodes and {len(dfs_edges)} edges.")
    if user_with_longest_tenure is not None:
        print(f"User with longest tenure in this cluster (found by DFS): {user_with_longest_tenure} (Tenure: {max_tenure})")
    else:
        print("Could not determine user with longest tenure (cluster might be too small).")
else:
    print("DFS could not be performed as no starting node was identified.")

In [ ]:
if selected_cluster_graph.nodes():
    plt.figure(figsize=(12, 12))
    pos_cluster = nx.spring_layout(selected_cluster_graph, seed=42) # Layout for the cluster

    # Draw all nodes in the cluster
    nx.draw_networkx_nodes(selected_cluster_graph, pos_cluster, node_color='lightgray', node_size=100)

    # Draw all edges in the cluster
    nx.draw_networkx_edges(selected_cluster_graph, pos_cluster, edge_color='lightgray', alpha=0.3)

    if start_node:
        # Highlight DFS nodes
        nx.draw_networkx_nodes(selected_cluster_graph, pos_cluster, nodelist=dfs_nodes, node_color='skyblue', node_size=150)

        # Highlight DFS edges
        nx.draw_networkx_edges(selected_cluster_graph, pos_cluster, edgelist=dfs_edges, edge_color='blue', width=1.5, alpha=0.7)

        # Highlight the starting node
        nx.draw_networkx_nodes(selected_cluster_graph, pos_cluster, nodelist=[start_node], node_color='green', node_size=200, label='DFS Start Node')

        if last_node is not None:
            nx.draw_networkx_nodes(selected_cluster_graph, pos_cluster, nodelist=[last_node], node_color='black', node_size=250, label='Last Node')
            nx.draw_networkx_labels(selected_cluster_graph, pos_cluster, labels={last_node: str(last_node)}, font_size=8, font_color='black')
        if start_node:
            # Dictionary to store distance from start_node
            # shortest_path_length gives us 'minimum' indirect distance.
            distances = nx.single_source_shortest_path_length(selected_cluster_graph, start_node)

            indirect_friends = [node for node, dist in distances.items() if dist > 3]
            f_of_f_of_f = [node for node, dist in distances.items() if dist == 3]
            friend_of_friend = [node for node, dist in distances.items() if dist == 2]
            direct_friends = [node for node, dist in distances.items() if dist == 1]

            print(f"User: {start_node}")
            print(f"Direct friends: {len(direct_friends)}")
            print(f"Indirect relations found: {len(indirect_friends)}")

            dfs_successors = list(nx.dfs_successors(selected_cluster_graph, start_node))
            # Hierarchy of the social chain:
            # Draw Direct Friends (Distance = 1)
            nx.draw_networkx_nodes(selected_cluster_graph, pos_cluster, nodelist=direct_friends, node_color='lightgreen', node_size=150, label='Direct Friends')
            # Draw Friends of Friends (Distance = 2)
            nx.draw_networkx_nodes(selected_cluster_graph, pos_cluster, nodelist=friend_of_friend, node_color='yellow', node_size=150, label='Friends of Friends')
            # Draw F of F of F (Distance = 3)
            nx.draw_networkx_nodes(selected_cluster_graph, pos_cluster, nodelist=f_of_f_of_f, node_color='orange', node_size=150, label='F. of F. Of Friends')
            # Draw Indirect Relations (Distance > 3)
            nx.draw_networkx_nodes(selected_cluster_graph, pos_cluster, nodelist=indirect_friends, node_color='red', node_size=150, label='Distant Relations')

    plt.title(f'DFS Traversal on Selected Cluster')
    plt.axis('off')
    plt.legend(scatterpoints=1)
    plt.show()
else:
    print("Cannot visualize an empty cluster.")

In [ ]:
if selected_cluster_graph.nodes():
    plt.figure(figsize=(12, 12))
    pos_cluster = nx.spring_layout(selected_cluster_graph, seed=42) # Layout for the cluster

    # Draw all nodes in the cluster
    nx.draw_networkx_nodes(selected_cluster_graph, pos_cluster, node_color='lightgray', node_size=100)

    # Draw all edges in the cluster
    nx.draw_networkx_edges(selected_cluster_graph, pos_cluster, edge_color='lightgray', alpha=0.3)

    if start_node:
        # Highlight DFS nodes
        nx.draw_networkx_nodes(selected_cluster_graph, pos_cluster, nodelist=dfs_nodes, node_color='skyblue', node_size=150)

        # Highlight DFS edges
        nx.draw_networkx_edges(selected_cluster_graph, pos_cluster, edgelist=dfs_edges, edge_color='blue', width=1.5, alpha=0.7)

        # Highlight the starting node
        nx.draw_networkx_nodes(selected_cluster_graph, pos_cluster, nodelist=[start_node], node_color='green', node_size=200, label='DFS Start Node')

        if last_node is not None:
            nx.draw_networkx_nodes(selected_cluster_graph, pos_cluster, nodelist=[last_node], node_color='black', node_size=250, label='Last Node')
            nx.draw_networkx_labels(selected_cluster_graph, pos_cluster, labels={last_node: str(last_node)}, font_size=8, font_color='black')
        if start_node:


            # Find nodes that act as bridges
            # bridges = list(nx.articulation_points(selected_cluster_graph)) <- Shortcut, but based on:
            # 1. Initialize data structures
            discovery_time = {node: -1 for node in selected_cluster_graph.nodes()}
            low_link = {node: -1 for node in selected_cluster_graph.nodes()}
            parent = {node: None for node in selected_cluster_graph.nodes()}
            articulation_points = set()
            timer = 0

            # 2. Manual DFS function with Articulation Point logic
            def find_bridges_manual(u):
                global timer
                discovery_time[u] = low_link[u] = timer
                timer += 1
                children = 0

                for v in selected_cluster_graph.neighbors(u):
                    if discovery_time[v] == -1:  # If v is not visited, it's a tree edge
                        parent[v] = u
                        children += 1
                        find_bridges_manual(v)

                        # Check if the subtree rooted at v has a connection back to u or above
                        low_link[u] = min(low_link[u], low_link[v])

                        # Case 1: u is root and has >1 children
                        if parent[u] is None and children > 1:
                            articulation_points.add(u)
                        # Case 2: u is not root and low value of child is >= discovery of u
                        if parent[u] is not None and low_link[v] >= discovery_time[u]:
                            articulation_points.add(u)

                    elif v != parent[u]:  # It's a back edge
                        low_link[u] = min(low_link[u], discovery_time[v])

            # 3. Run it from your start node
            if start_node:
                find_bridges_manual(start_node)
                print(f"Bridge Nodes: {articulation_points}")
                nx.draw_networkx_nodes(selected_cluster_graph, pos_cluster,
                                      nodelist=list(articulation_points),
                                      node_color='blue',
                                      node_size=300,
                                      label='Bridge Nodes')

    plt.title(f'DFS Traversal on Selected Cluster')
    plt.axis('off')
    plt.legend(scatterpoints=1)
    plt.show()
else:
    print("Cannot visualize an empty cluster.")